In [1]:
!pip install anndata=='0.8.0'

  Using cached anndata-0.8.0-py3-none-any.whl (96 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 10.6 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: h5py
    Found existing installation: h5py 2.10.0
    Uninstalling h5py-2.10.0:
      Successfully uninstalled h5py-2.10.0
  Attempting uninstall: anndata
    Found existing installation: anndata 0.7.8
    Uninstalling anndata-0.7.8:
      Successfully uninstalled anndata-0.7.8
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
samap 1.0.2 requires h5py<=2.10, but you have h5py 3.8.0 which is incompatible.
sam-algorithm 1.0.0 requires h5py<=2.10.0, but you have h5py 3.8.0 which is incompatible.


In [2]:
!pip install openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.3/251.3 kB 3.7 MB/s eta 0:00:0000:01


In [1]:
from samalg import SAM
import scanpy as sc
import statistics
import matplotlib.pyplot as plt
import seaborn as sns
import random
import pandas as pd
import matplotlib.colors
import scipy
import numpy as np
import seaborn as sns 
from tqdm import tqdm
import matplotlib.colors as mcolors
import pickle
import matplotlib as mpl

/scratch/miniconda/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open('parent_dict.pkl', 'rb') as f:
    parent_dict = pickle.load(f)

In [3]:
dat = pd.read_csv('MERFISH_data/C57BL6_ABC_cell_metadata_with_parcellation_annotation.csv', index_col = 'cell_label')

In [4]:
for item in dat['subclass'].unique():
    if item not in parent_dict:
        parent_dict[item] = 'not hypo'

In [5]:
parent_dict['not hypo'] = 'not hypo'

In [8]:
sam = SAM()
sam.load_data('Active_SAM_joined/SAM_DR_ncbi_joined_cleaned_07172026.h5ad')

In [10]:
sam.adata.obs.columns

Index(['n_genes', 'n_counts', 'key', 'leiden_clusters_neuron', 'eq_subclass',
       'eq_subclass_lc', 'leiden_clusters', 'eq_subclass_nounlabeled',
       'ss_subclass', 'ss_subclass_v4_nounlabeled',
       'ss_subclass_v4_nounlabeled_nn', 'ss_subclass_nounlabeled_nmm_v4_nn',
       'ss_subclass_nounlabeled_nmm_v4_nn_thresh30',
       'ss_subclass_nounlabeled_nmm_cl_v4_nn', 'neurotransmitter_v3'],
      dtype='object')

In [16]:
metadata = sam.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'].value_counts().to_frame()

In [17]:
metadata.columns = ['# Cells']

In [18]:
metadata

,# Cells
cj_ac_xt_dr_1,5329
cj_ac_xt_dr_4,4264
dr_2,3768
cj_xt_dr_1,2863
cj_xt_dr_2,2816
...,...
130 LHA Pmch Glut,77
054 STR Prox1 Lhx6 Gaba,77
049 Lamp5 Gaba,69
062 STR D2 Gaba,66


In [19]:
organism = 'dr'

In [20]:
hl = []
for item in dat['subclass'].unique():
    if item in parent_dict:
        if parent_dict[parent_dict[item]] == 'hypo':
            hl.append(item)

In [21]:
dat.columns

Index(['brain_section_label', 'cluster_alias', 'average_correlation_score',
       'feature_matrix_label', 'donor_label', 'donor_genotype', 'donor_sex',
       'x_section', 'y_section', 'z_section', 'neurotransmitter', 'class',
       'subclass', 'supertype', 'cluster', 'neurotransmitter_color',
       'class_color', 'subclass_color', 'supertype_color', 'cluster_color',
       'x_reconstructed', 'y_reconstructed', 'z_reconstructed',
       'parcellation_index', 'x_ccf', 'y_ccf', 'z_ccf', 'parcellation_organ',
       'parcellation_category', 'parcellation_division',
       'parcellation_structure', 'parcellation_substructure',
       'parcellation_organ_color', 'parcellation_category_color',
       'parcellation_division_color', 'parcellation_structure_color',
       'parcellation_substructure_color'],
      dtype='object')

In [22]:
not_hypo_list = []
for item in dat['subclass'].unique():
    if parent_dict[parent_dict[item]] == 'not hypo' and item[-2:] != 'NN':
        not_hypo_list.append(item)
nonhypo_dat = dat[dat['subclass'].isin(not_hypo_list)]

In [23]:
metadata.sort_values('# Cells', ascending = False).head(20)

,# Cells
cj_ac_xt_dr_1,5329
cj_ac_xt_dr_4,4264
dr_2,3768
cj_xt_dr_1,2863
cj_xt_dr_2,2816
dr_5,2810
339 Astrocyte-like NN,2748
dr_7,2216
338 Lymphoid NN,2216
dr_8,2170


In [24]:
np.sort(dat['brain_section_label'].unique())

array(['C57BL6J-638850.05', 'C57BL6J-638850.06', 'C57BL6J-638850.08',
       'C57BL6J-638850.09', 'C57BL6J-638850.10', 'C57BL6J-638850.11',
       'C57BL6J-638850.12', 'C57BL6J-638850.13', 'C57BL6J-638850.14',
       'C57BL6J-638850.15', 'C57BL6J-638850.16', 'C57BL6J-638850.17',
       'C57BL6J-638850.18', 'C57BL6J-638850.19', 'C57BL6J-638850.24',
       'C57BL6J-638850.25', 'C57BL6J-638850.26', 'C57BL6J-638850.27',
       'C57BL6J-638850.28', 'C57BL6J-638850.29', 'C57BL6J-638850.30',
       'C57BL6J-638850.31', 'C57BL6J-638850.32', 'C57BL6J-638850.33',
       'C57BL6J-638850.35', 'C57BL6J-638850.36', 'C57BL6J-638850.37',
       'C57BL6J-638850.38', 'C57BL6J-638850.39', 'C57BL6J-638850.40',
       'C57BL6J-638850.42', 'C57BL6J-638850.43', 'C57BL6J-638850.44',
       'C57BL6J-638850.45', 'C57BL6J-638850.46', 'C57BL6J-638850.47',
       'C57BL6J-638850.48', 'C57BL6J-638850.49', 'C57BL6J-638850.50',
       'C57BL6J-638850.51', 'C57BL6J-638850.52', 'C57BL6J-638850.54',
       'C57BL6J-6388

In [ ]:
for i in range(25,65):
    section = 'C57BL6J-638850.' + str(i)
    sec = dat[dat['brain_section_label'] == section]
    nonhypo_sec = nonhypo_dat[nonhypo_dat['brain_section_label'] == section]
    hy_cells = sec[sec['parcellation_division'] == 'HY']
    
    nonhyposec_ct = nonhypo_sec[nonhypo_sec['subclass'].isin(metadata.index)]
    nc = []
    mapping = []
    for item in nonhyposec_ct['subclass']:
            nc.append(metadata.loc[item,'# Cells'])
    nonhyposec_ct['NumCells'] = nc
    
    #####Uncomment
    plt.rcParams["figure.figsize"] = (13,7)
    legned = True
    fig, ax = plt.subplots()
    vmin = 0
    vmax = 2000
    norm = mcolors.Normalize(vmin, vmax)
    g = sns.scatterplot(data = sec, x = 'z_ccf',y = 'y_ccf', s = 5, color = 'lightgrey', edgecolor = 'None')
    g_one = sns.scatterplot(data = nonhyposec_ct, x = 'z_ccf', y = 'y_ccf', hue = 'NumCells', s = 5, alpha = 1, palette = 'Blues',hue_norm=norm, edgecolor = 'None')
    g_two = sns.scatterplot(data = hy_cells, x = 'z_ccf',y = 'y_ccf', s = 5, alpha = .5, color = '#dd9396', edgecolor = 'None')
    ax.axis('off')
    sm = mpl.cm.ScalarMappable(cmap="Blues", norm=norm)
    sm.set_array([])
    ax.legend([])
    plt.colorbar(sm, ax=ax, label="Number of cells")
    plt.savefig('Figures/Figures_07242026/DR_MERFISH_nonhypo/C57B6_ABC_' + organism + '_' + section + '_nonhypoblue_numcells_colorbar_07312026.png', dpi = 300)
    plt.show()
    plt.clf()